# Phase 9 Lab — Reference Solution

**Phase:** Deep Learning  
**Scenario:** A research team needs reproducible neural baselines for tabular and image tasks before considering larger architectures.

**Deliverable:** A training notebook with shape checks, tiny-batch overfit test, baseline comparison, checkpoints, and error analysis.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Specify tensor shapes and task losses.
2. Create deterministic train/validation/test partitions.
3. Implement correct training/evaluation loops.
4. Overfit a tiny batch as a pipeline test.
5. Train an MLP and CNN with early-stopping logic.
6. Inspect gradients and class-wise errors.
7. Record reproducibility and serving requirements.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(42)

X=np.load(DATA_DIR/"images/simple_shapes_X.npy")
y=np.load(DATA_DIR/"images/simple_shapes_y.npy")
order=np.random.default_rng(42).permutation(len(X))
train_idx,test_idx=order[:380],order[380:]
loader=DataLoader(TensorDataset(torch.tensor(X[train_idx]),torch.tensor(y[train_idx])),
                  batch_size=48,shuffle=True)
Xtest=torch.tensor(X[test_idx]); ytest=torch.tensor(y[test_idx])
model=nn.Sequential(
    nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
    nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
    nn.Flatten(),nn.Linear(16*4*4,3)
)
optimizer=torch.optim.AdamW(model.parameters(),lr=.004,weight_decay=1e-4)
loss_fn=nn.CrossEntropyLoss()
history=[]
for epoch in range(12):
    model.train(); total=0
    for xb,yb in loader:
        optimizer.zero_grad()
        loss=loss_fn(model(xb),yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        optimizer.step()
        total+=loss.item()*len(xb)
    model.eval()
    with torch.no_grad():
        accuracy=(model(Xtest).argmax(1)==ytest).float().mean().item()
    history.append((epoch+1,total/len(train_idx),accuracy))
display(pd.DataFrame(history,columns=["epoch","train_loss","test_accuracy"]))
torch.save({"model_state":model.state_dict(),"classes":["vertical","horizontal","diagonal"]},
           ARTIFACT_DIR/"phase9_shapes_cnn.pt")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.